<a href="https://colab.research.google.com/github/stylekkm051049-bit/damo/blob/main/%E0%B8%A3%E0%B8%B0%E0%B8%9A%E0%B8%9A%E0%B9%82%E0%B8%A3%E0%B8%87%E0%B9%80%E0%B8%A3%E0%B8%B5%E0%B8%A2%E0%B8%99%E0%B8%81%E0%B8%A7%E0%B8%94%E0%B8%A7%E0%B8%B4%E0%B8%8A%E0%B8%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ระบบลงทะเบียนคอร์สเรียนโรงเรียนกวดวิชา
### Coding Challenge — หัวข้อที่ 11

**องค์กร:** โรงเรียนกวดวิชา  
**ระบบ:** ระบบลงทะเบียนคอร์สเรียน

ระบบจำลองนี้ออกแบบตามรายละเอียดของกลุ่ม โดยมีนักเรียน (Student) เป็นผู้เริ่มต้นกระบวนการ นักเรียนเลือกคอร์ส (Course) จากนั้นระบบตรวจสอบจำนวนที่นั่ง หากยังมีที่ว่างจะสร้างรายการลงทะเบียน (Enrollment) คำนวณค่าเรียนรวม และบันทึกการชำระเงิน (Payment)

ข้อมูลหลักของระบบแบ่งเป็น 4 ส่วน ได้แก่ Student, Course, Enrollment และ Payment โดย Enrollment ทำหน้าที่เชื่อมข้อมูลระหว่าง Student และ Course

##ส่วนที่ 1 - ขั้นตอนการทำงานของระบบ
นักเรียน 1 คนสามารถเลือกได้ 1 คอร์สต่อการลงทะเบียน 1 ครั้ง โดยระบบมีขั้นตอนการทำงานดังนี้
**นักเรียนเลือกคอร์ส → ตรวจสอบที่นั่ง → ลงทะเบียน → คำนวณค่าเรียน → ชำระเงิน → จัดเก็บข้อมูล → วิเคราะห์ข้อมูล**

### ข้อมูลหลัก
- **Student:** รหัสนักเรียน ชื่อ เพศ อายุ ระดับชั้น โรงเรียน และเบอร์โทรศัพท์
- **Course:** รหัสคอร์ส ชื่อคอร์ส วิชา ครูผู้สอน ราคา จำนวนที่นั่ง และจำนวนผู้สมัคร
- **Enrollment:** รหัสการลงทะเบียน รหัสนักเรียน รหัสคอร์ส วันที่ลงทะเบียน ส่วนลด ค่าเรียนรวม และสถานะ
- **Payment:** รหัสการชำระเงิน รหัสการลงทะเบียน วันที่ชำระ จำนวนเงิน วิธีชำระ และสถานะ

ในงานนี้จะจำลองข้อมูลนักเรียน 450 คน และจำลองการลงทะเบียน 450 รายการ โดยใช้ for loop และ random เพื่อจำลองการทำงานของระบบทีละรายการ จากนั้นแปลงข้อมูลเป็น DataFrame และบันทึกเป็นไฟล์ CSV และฐานข้อมูล SQLite เพื่อนำไปวิเคราะห์ข้อมูลด้วย pandas และ SQL

## ส่วนที่ 2 — เตรียม class
ระบบประกอบด้วย 4 Class หลัก ได้แก่ `Student`, `Course`, `Enrollment` และ `Payment`

แต่ละ Class มี Attribute สำหรับเก็บข้อมูล และ Method สำหรับการทำงานของระบบจริง

In [1]:
import random
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

random.seed(11)
print("เตรียม Library เรียบร้อยแล้ว")

เตรียม Library เรียบร้อยแล้ว


In [2]:
class Student:
    def __init__(self, student_id, name, gender, age, grade, school, phone):
        self.student_id = student_id
        self.name = name
        self.gender = gender
        self.age = age
        self.grade = grade
        self.school = school
        self.phone = phone

    def get_grade(self):
        return self.grade

class Course:
    def __init__(self, course_id, course_name, subject, teacher, price, capacity):
        self.course_id = course_id
        self.course_name = course_name
        self.subject = subject
        self.teacher = teacher
        self.price = price
        self.capacity = capacity
        self.enrolled_count = 0

    def has_seat(self):
        return self.enrolled_count < self.capacity

    def add_student(self):
        if self.has_seat():
            self.enrolled_count += 1
            return True
        return False

    def available_seats(self):
        return self.capacity - self.enrolled_count


class Enrollment:
    def __init__(
        self,
        enrollment_id,
        student,
        course,
        enroll_date,
        discount_rate=0
    ):
        self.enrollment_id = enrollment_id
        self.student = student
        self.course = course
        self.enroll_date = enroll_date
        self.discount_rate = discount_rate
        self.total_tuition = 0
        self.status = "รอตรวจสอบ"

    def calculate_total(self):
        discount = self.course.price * self.discount_rate
        self.total_tuition = round(
            self.course.price - discount,
            2
        )
        return self.total_tuition

    def confirm(self):
        if self.course.add_student():
            self.status = "ลงทะเบียนสำเร็จ"
            self.calculate_total()
            return True

        self.status = "คอร์สเต็ม"
        return False


class Payment:
    def __init__(
        self,
        payment_id,
        enrollment,
        payment_date,
        method
    ):
        self.payment_id = payment_id
        self.enrollment = enrollment
        self.payment_date = payment_date
        self.amount = enrollment.total_tuition
        self.method = method
        self.status = "ชำระเงินแล้ว"

    def record_payment(self):
        return self.status


print("สร้าง Class ทั้ง 4 Class สำเร็จ")

สร้าง Class ทั้ง 4 Class สำเร็จ
